# Lab 02 · Parte 3 — Principio multiplicativo: outfits de una tienda online (Python)

Una tienda de moda online quiere generar recomendaciones automáticas de **outfits completos**: una camiseta, un pantalón y un par de zapatos.

| Categoría | Prendas | Opciones |
|---|---|---|
| Camisetas | C1, C2, C3 | 3 |
| Pantalones | P1, P2 | 2 |
| Zapatos | Z1, Z2 | 2 |

Un outfit es un elemento del **producto cartesiano** $C \times P \times Z$: una terna ordenada (camiseta, pantalón, zapatos).

In [1]:
from itertools import product
from math import prod

import pandas as pd

prendas = {
    "camiseta": ["C1", "C2", "C3"],
    "pantalon": ["P1", "P2"],
    "zapatos": ["Z1", "Z2"],
}

## Parte 1. Principio multiplicativo

### 1–2. Cálculo paso a paso

El **principio multiplicativo** dice que si una elección se hace en varias etapas independientes, el número total de resultados es el **producto** de las opciones de cada etapa.

1. **Elegir la camiseta:** hay **3** opciones.
2. **Elegir el pantalón:** para *cada* camiseta hay **2** pantalones, así que hay $3 \times 2 = 6$ combinaciones camiseta + pantalón.
3. **Elegir los zapatos:** para *cada* una de esas 6 combinaciones hay **2** pares de zapatos, así que hay $6 \times 2 = 12$ outfits.

$$|C \times P \times Z| = |C| \cdot |P| \cdot |Z| = 3 \cdot 2 \cdot 2 = 12$$

El diagrama de árbol de la celda siguiente muestra las tres etapas: cada nivel multiplica las ramas del anterior.

In [2]:
opciones = {categoria: len(lista) for categoria, lista in prendas.items()}
total_teorico = prod(opciones.values())

acumulado = 1
for paso, (categoria, n) in enumerate(opciones.items(), start=1):
    acumulado *= n
    print(f"Paso {paso}: elegir {categoria:<8} -> {n} opciones -> {acumulado} combinaciones")

print(f"\nTotal: {' x '.join(map(str, opciones.values()))} = {total_teorico} outfits")

Paso 1: elegir camiseta -> 3 opciones -> 3 combinaciones
Paso 2: elegir pantalon -> 2 opciones -> 6 combinaciones
Paso 3: elegir zapatos  -> 2 opciones -> 12 combinaciones

Total: 3 x 2 x 2 = 12 outfits


In [3]:
# Diagrama de árbol: cada nivel es una etapa de la elección
for c in prendas["camiseta"]:
    print(c)
    for i, p in enumerate(prendas["pantalon"]):
        rama_p = "└──" if i == len(prendas["pantalon"]) - 1 else "├──"
        print(f"  {rama_p} {p}")
        for j, z in enumerate(prendas["zapatos"]):
            rama_z = "└──" if j == len(prendas["zapatos"]) - 1 else "├──"
            sangria = "    " if i == len(prendas["pantalon"]) - 1 else "│   "
            print(f"  {sangria}{rama_z} {z}  ->  ({c}, {p}, {z})")

C1
  ├── P1
  │   ├── Z1  ->  (C1, P1, Z1)
  │   └── Z2  ->  (C1, P1, Z2)
  └── P2
      ├── Z1  ->  (C1, P2, Z1)
      └── Z2  ->  (C1, P2, Z2)
C2
  ├── P1
  │   ├── Z1  ->  (C2, P1, Z1)
  │   └── Z2  ->  (C2, P1, Z2)
  └── P2
      ├── Z1  ->  (C2, P2, Z1)
      └── Z2  ->  (C2, P2, Z2)
C3
  ├── P1
  │   ├── Z1  ->  (C3, P1, Z1)
  │   └── Z2  ->  (C3, P1, Z2)
  └── P2
      ├── Z1  ->  (C3, P2, Z1)
      └── Z2  ->  (C3, P2, Z2)


### 3. Interpretación en un sistema de recomendación

- El catálogo de **7 prendas** genera **12 outfits distintos**: el sistema tiene 12 recomendaciones posibles, casi el doble de opciones que prendas.
- El número de outfits **crece multiplicando, no sumando**: si se añade una cuarta camiseta, los outfits pasan de 12 a $4 \cdot 2 \cdot 2 = 16$, sin tocar el resto del catálogo. Con un catálogo real (por ejemplo, 200 camisetas, 80 pantalones y 50 zapatos) serían 800 000 combinaciones.
- Por eso un recomendador real **no puede enseñarlas todas**: necesita filtrar y ordenar (por estilo, colores, talla, stock o historial del cliente). Saber cuántas combinaciones hay permite estimar el coste de evaluarlas todas antes de diseñar el sistema.

## Parte 2. Generación de combinaciones

### 1. Generar todos los outfits

`itertools.product` calcula el producto cartesiano de las tres listas: devuelve todas las ternas (camiseta, pantalón, zapatos).

In [4]:
outfits = list(product(prendas["camiseta"], prendas["pantalon"], prendas["zapatos"]))

for numero, outfit in enumerate(outfits, start=1):
    print(f"{numero:>2}. {outfit}")

 1. ('C1', 'P1', 'Z1')
 2. ('C1', 'P1', 'Z2')
 3. ('C1', 'P2', 'Z1')
 4. ('C1', 'P2', 'Z2')
 5. ('C2', 'P1', 'Z1')
 6. ('C2', 'P1', 'Z2')
 7. ('C2', 'P2', 'Z1')
 8. ('C2', 'P2', 'Z2')
 9. ('C3', 'P1', 'Z1')
10. ('C3', 'P1', 'Z2')
11. ('C3', 'P2', 'Z1')
12. ('C3', 'P2', 'Z2')


### 2. Representación en tabla

In [5]:
tabla = pd.DataFrame(outfits, columns=["camiseta", "pantalon", "zapatos"])
tabla.index = [f"Outfit {i}" for i in range(1, len(tabla) + 1)]
tabla

,camiseta,pantalon,zapatos
Outfit 1,C1,P1,Z1
Outfit 2,C1,P1,Z2
Outfit 3,C1,P2,Z1
Outfit 4,C1,P2,Z2
Outfit 5,C2,P1,Z1
Outfit 6,C2,P1,Z2
Outfit 7,C2,P2,Z1
Outfit 8,C2,P2,Z2
Outfit 9,C3,P1,Z1
Outfit 10,C3,P1,Z2


### 3. Verificación

El número de combinaciones generadas debe coincidir con el principio multiplicativo, y ninguna debe repetirse.

In [6]:
print(f"Combinaciones generadas : {len(outfits)}")
print(f"Combinaciones distintas : {len(set(outfits))}")
print(f"Principio multiplicativo: {total_teorico}")

assert len(outfits) == len(set(outfits)) == total_teorico
print("\nVerificación correcta: coinciden.")

Combinaciones generadas : 12
Combinaciones distintas : 12
Principio multiplicativo: 12

Verificación correcta: coinciden.


In [7]:
# Cada prenda aparece en tantos outfits como combinaciones haya del resto de categorías
(tabla
 .melt(var_name="categoria", value_name="prenda")
 .value_counts(["categoria", "prenda"], sort=False)
 .rename("outfits")
 .to_frame())

outfits
categoria prenda         
camiseta  C1            4
          C2            4
          C3            4
pantalon  P1            6
          P2            6
zapatos   Z1            6
          Z2            6

## Conclusiones

- Con 3 camisetas, 2 pantalones y 2 pares de zapatos se forman **12 outfits** ($3 \cdot 2 \cdot 2$).
- La generación exhaustiva produce exactamente **12 combinaciones distintas**, lo que verifica el principio multiplicativo.
- Cada prenda aparece en tantos outfits como combinaciones haya del resto de categorías: cada camiseta en $2 \cdot 2 = 4$ outfits y cada pantalón o par de zapatos en $3 \cdot 2 = 6$.